# Phase 1 — agreement studies (working notebook)

Foundations for the Phase 1 write-up (`docs/phase1-design.md`): the effectiveness gate,
the RQ1 lexical agreement profiles, the gap-gradient validity control, the joint fit,
and the RQ2 WordNet-tier semantic axioms. Built on whatever grid cells have landed in
`results/` — rerun the load cell as more arrive.

**Grid**: {DL19, DL20} × {top-10 all-pairs (*primary*), uniform depth-100 (*validity
control*)} × {Qwen 3.6 35B (primary, prompt v1), Flan-T5-large (contrast, v0)}.
All axiom preferences below postdate the 2026-07-11 `ir_axioms` PROX fixes
(PROX2 sign flip, PROX1 determinism) — unlike the Phase 0 pilot numbers.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from axiomrank.paths import results_dir

INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASE, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
PRIMARY = "models/qwen3.6-35B-A3B-AWQ"
CONTRAST = "google/flan-t5-large"
MODEL_COLOR = {PRIMARY: "#2a78d6", CONTRAST: "#1baf7a"}
MODEL_LABEL = {PRIMARY: "Qwen 3.6 35B", CONTRAST: "Flan-T5-large"}
COND_COLOR = {"top10": "#2a78d6", "uniform": "#4a3aa7"}
CELLS = ["dl19_top10", "dl19_uniform", "dl20_top10", "dl20_uniform"]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": INK2,
    "axes.edgecolor": BASE, "font.size": 11,
})


def load_cells(experiment: str, filename: str) -> dict:
    """{(cell, model): DataFrame|dict} for every grid cell that has `filename`."""
    out = {}
    for path in sorted(results_dir(experiment).glob(f"dl*/metrics/*/{filename}")):
        cell, _, model_dir = path.relative_to(results_dir(experiment)).parts[:3]
        model = model_dir.replace("__", "/")
        out[(cell, model)] = (
            pd.read_csv(path) if filename.endswith(".csv") else json.loads(path.read_text())
        )
    return out


rq1 = load_cells("rq1_lexical_agreement", "agreement.csv")
rq1_gap = load_cells("rq1_lexical_agreement", "gap_agreement.csv")
rq1_fit = load_cells("rq1_lexical_agreement", "joint_fit.json")
rq2 = load_cells("rq2_semantic_agreement", "agreement.csv")
rq2_fit = load_cells("rq2_semantic_agreement", "joint_fit.json")
eff = load_cells("ranking_effectiveness", "effectiveness.json")

status = pd.DataFrame(
    {
        f"{label} · {MODEL_LABEL[m]}": {
            cell: "✓" if (cell, m) in data else "—" for cell in CELLS
        }
        for label, data in [("RQ1", rq1), ("RQ2", rq2), ("gate", eff)]
        for m in [PRIMARY, CONTRAST]
    }
)
status  # gate is only defined on the top-10 condition

## 1. Effectiveness gate — the top-10 residual is skill

Copeland aggregation of the cached pairwise verdicts (= PRP-allpair) vs the BM25
first-stage run, nDCG@10 against TREC qrels. **Gate: Qwen must clearly beat BM25 on
both collections** (design §4); Flan-T5 is contrast only.

In [ ]:
rows = []
for (cell, model), e in sorted(eff.items()):
    m = e["metrics"]["nDCG@10"]
    rows.append({
        "collection": cell.split("_")[0].upper(), "model": model,
        "bm25": m["mean_baseline"], "reranked": m["mean_reranked"],
        "delta": m["mean_delta"], "wtl": f'{m["wins"]}/{m["ties"]}/{m["losses"]}',
    })
gate = pd.DataFrame(rows).sort_values(["collection", "model"], ascending=[True, False])

fig, ax = plt.subplots(figsize=(7.5, 2.9))
for y, r in enumerate(gate.itertuples()):
    ax.plot([r.bm25, r.reranked], [y, y], color=GRID, linewidth=2, zorder=1)
    ax.scatter(r.bm25, y, s=60, color=MUTED, zorder=2)
    ax.scatter(r.reranked, y, s=72, color=MODEL_COLOR[r.model], zorder=3)
    ax.text(r.reranked + 0.006, y, f"+{r.delta:.3f}   W/T/L {r.wtl}",
            va="center", fontsize=9.5, color=INK2)
ax.set_yticks(range(len(gate)),
              [f"{r.collection} · {MODEL_LABEL[r.model]}" for r in gate.itertuples()])
ax.invert_yaxis()
ax.set_xlim(0.45, 0.66)
ax.set_xlabel("nDCG@10  (grey = BM25 baseline, colored = Copeland-reranked)")
ax.set_title("Effectiveness gate: PRP-allpair vs BM25", loc="left", fontsize=12)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(length=0)
plt.tight_layout()
plt.show()

**Gate passed.** Qwen beats BM25 by +0.069 (DL19) and +0.062 (DL20) nDCG@10 with
lopsided win/loss counts (32/6, 41/9) — the residual Phase 1 studies is genuine
reranking skill. Flan-T5 also clears BM25 (pairwise aggregation is forgiving of a
noisy judge) but trails Qwen everywhere. Reranked ≈ 0.55 sits below the ≈ 0.65
literature anchor for strong PRP rerankers — worth a sentence in the write-up, not a
gate failure.

## 2. RQ1 — per-axiom agreement profiles (Qwen)

Agreement with the judge where the axiom fires and the judge is decisive, with
query-bootstrap 95% CIs. Blue = top-10 all-pairs (primary), violet = uniform
depth-100 control. TFC3 and M-TDC are omitted (they fire on < 15 pairs per cell);
relaxed-precondition variants are in §5.

In [ ]:
MIN_N = 30
base = [a for a in rq1[("dl19_top10", PRIMARY)].axiom
        if "@" not in a and a not in ("TFC3", "M_TDC")]
order = (pd.concat([rq1[(c, PRIMARY)].set_index("axiom").agreement for c in CELLS
                    if (c, PRIMARY) in rq1], axis=1)
         .loc[base].mean(axis=1).sort_values().index.tolist())

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.6), sharey=True)
for ax, coll in zip(axes, ["dl19", "dl20"]):
    ax.axvline(0.5, color=MUTED, linewidth=1, linestyle=(0, (4, 3)))
    for cond, dy in [("top10", 0.16), ("uniform", -0.16)]:
        key = (f"{coll}_{cond}", PRIMARY)
        if key not in rq1:
            continue
        df = rq1[key].set_index("axiom").loc[order]
        ok = df.n_evaluable >= MIN_N
        y = [order.index(a) + dy for a in df.index]
        ax.errorbar(df.agreement[ok], [v for v, k in zip(y, ok) if k],
                    xerr=[(df.agreement - df.ci_lo)[ok], (df.ci_hi - df.agreement)[ok]],
                    fmt="o", ms=6, color=COND_COLOR[cond], ecolor=GRID,
                    elinewidth=1.6, capsize=0, label=cond, zorder=3)
        ax.scatter(df.agreement[~ok], [v for v, k in zip(y, ok) if not k],
                   s=36, color=GRID, zorder=2)
    ax.set_title(coll.upper(), loc="left", fontsize=11, color=INK2)
    ax.set_xlim(0.28, 1.0)
    ax.set_xlabel("agreement with judge")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.xaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)
axes[0].set_yticks(range(len(order)), order)
axes[0].legend(loc="lower right", frameon=False, title=None)
fig.suptitle("RQ1 agreement profile, Qwen (grey = n < 30; bars = 95% bootstrap CI)",
             x=0.02, ha="left", fontsize=12)
plt.tight_layout()
plt.show()

The profile is consistent across collections: the best-aligned axioms are the
document-structure ones — **AND (~0.65–0.80) and LB1 (~0.73–0.77)** — not the classic
TF core. **TFC1 sits at chance** in every cell (top-10 *and* uniform), and **DIV is at
or below chance** despite firing on > 90% of pairs. The proximity family lands in
between (~0.55–0.68). Note the CIs: with 43–54 queries most axioms' intervals span
±0.05–0.10, so small cross-cell differences should not be over-read.

## 3. Gap gradient — the validity control

Design §2.3 expects agreement near chance on adjacent-rank pairs, rising with BM25
rank gap. Uniform cells, Qwen; x = upper edge of the gap decile.

In [ ]:
series = [("AND", "#2a78d6"), ("TFC1", "#1baf7a"), ("DIV", "#eda100")]

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), sharey=True)
for ax, coll in zip(axes, ["dl19", "dl20"]):
    key = (f"{coll}_uniform", PRIMARY)
    if key not in rq1_gap:
        ax.set_axis_off()
        continue
    g = rq1_gap[key]
    bins = g.groupby("gap_bin")[["decisive_rate"]].first()
    ax.axhline(0.5, color=MUTED, linewidth=1, linestyle=(0, (4, 3)))
    ax.plot(bins.index, bins.decisive_rate, color=MUTED, linewidth=1.6,
            linestyle=(0, (1, 2)), marker=".", label="judge decisive rate")
    for name, c in series:
        s = g[g.axiom == name].set_index("gap_bin").agreement
        ax.plot(s.index, s, color=c, linewidth=2, marker="o", ms=4, label=name)
        ax.annotate(name, (s.index[-1], s.iloc[-1]), xytext=(5, 0),
                    textcoords="offset points", color=c, fontsize=9.5, va="center")
    ax.set_title(coll.upper(), loc="left", fontsize=11, color=INK2)
    ax.set_xlabel("BM25 rank gap (decile upper edge)")
    ax.set_xlim(0, 112)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)
axes[0].set_ylabel("agreement / rate")
axes[0].legend(loc="upper left", frameon=False, fontsize=9)
fig.suptitle("Agreement vs BM25 rank gap (uniform condition, Qwen)",
             x=0.02, ha="left", fontsize=12)
plt.tight_layout()
plt.show()

The gradient is **weaker than the design expected**. The judge's decisive rate rises
cleanly with gap (~0.55 → ~0.70–0.75), and TFC1 climbs only in the widest deciles
(~0.6–0.7 at gap ≈ 100). But there is no broad "classical axioms visibly work on
wide-gap pairs" regime: AND is high at *every* gap, and DIV drifts **below** chance as
the gap widens (0.55 → 0.33–0.36) — on easy pairs the judge systematically prefers the
document DIV penalises. Design §2.3 says a missing gradient is a finding to chase
before RQ3, not to skip; this needs its own look (per-axiom gradients + CIs).

## 4. Joint fit — how much of the judge do the axioms explain?

Query-grouped 5-fold CV logistic regression on decisive pairs: strict core (10
classical axioms) vs full battery (+ AND/DIV/LB1 + relaxed variants). Baseline is the
majority class rate.

In [ ]:
rows = []
for cell in CELLS:
    if (cell, PRIMARY) not in rq1_fit:
        continue
    jf = rq1_fit[(cell, PRIMARY)]
    rows.append({
        "cell": cell,
        "base rate": jf["strict_core"]["base_rate"],
        "strict core": jf["strict_core"]["cv_accuracy"],
        "full battery": jf["full_battery"]["cv_accuracy"],
        "AUC (full)": jf["full_battery"]["cv_auc"],
        "n pairs": jf["strict_core"]["n_decisive_pairs"],
    })
fit = pd.DataFrame(rows).set_index("cell")

fig, ax = plt.subplots(figsize=(7.5, 2.9))
for y, (cell, r) in enumerate(fit.iterrows()):
    ax.plot([r["base rate"], r["full battery"]], [y, y], color=GRID, lw=2, zorder=1)
    ax.scatter(r["base rate"], y, s=60, color=MUTED, zorder=2, label="base rate" if y == 0 else None)
    ax.scatter(r["strict core"], y, s=64, color="#2a78d6", zorder=3, label="strict core" if y == 0 else None)
    ax.scatter(r["full battery"], y, s=64, color="#4a3aa7", zorder=3, label="full battery" if y == 0 else None)
ax.set_yticks(range(len(fit)), fit.index)
ax.invert_yaxis()
ax.set_xlabel("CV accuracy on decisive pairs")
ax.set_title("Joint fit vs majority-class baseline (Qwen)", loc="left", fontsize=12)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(length=0)
ax.legend(loc="lower right", frameon=False, fontsize=9)
plt.tight_layout()
fit.round(3)

## 5. RQ2 — do WordNet-tier semantic axioms add anything?

STMC1/STMC2/REG/ANTI-REG with the WordNet similarity backend, and the design §2.5
question: does adding them to the lexical battery improve the joint fit?

In [ ]:
sem = pd.concat({cell: rq2[(cell, PRIMARY)].set_index("axiom").loc[
                     lambda d: d.index.str.endswith("@wn"),
                     ["coverage", "n_evaluable", "agreement", "ci_lo", "ci_hi"]]
                 for cell in CELLS if (cell, PRIMARY) in rq2}, names=["cell"])
display(sem.round(3))

delta = pd.DataFrame({
    cell: {
        "lexical cv_acc": jf["lexical"]["cv_accuracy"],
        "combined cv_acc": jf["combined"]["cv_accuracy"],
        "Δ accuracy": jf["combined"]["cv_accuracy"] - jf["lexical"]["cv_accuracy"],
        "Δ AUC": jf["combined"]["cv_auc"] - jf["lexical"]["cv_auc"],
    }
    for cell in CELLS if (cell, PRIMARY) in (locals().get("rq2_fit") or rq2_fit)
    for jf in [rq2_fit[(cell, PRIMARY)]]
}).T
delta.round(3)

## Takeaways so far & open items

1. **Gate passed** — the top-10 residual is skill; every downstream analysis is licensed.
2. **The headline null replicates on DL20**: the classical TF core explains almost
   nothing (TFC1 at chance everywhere); the alignment that *does* exist lives in
   AND/LB1 and, more weakly, proximity.
3. **The gap gradient did not appear as designed** — judge decisiveness rises with gap
   but per-axiom agreement mostly doesn't (and DIV inverts). This is the §2.3
   "finding to chase before RQ3".
4. **WordNet semantics add no joint predictive power** (combined ≤ lexical on cv
   accuracy and AUC) — evidence for a no-go on the 7.24 GB fastText tier unless the
   per-axiom numbers argue otherwise.

**Still missing / TODO**
- Flan-T5 cells: `dl20_uniform` (RQ1, collecting as of 2026-07-12) and `dl20_top10` (RQ2).
- Relaxed-variant section (§5 of the design): margins are in `agreement.csv`
  (`AXIOM@lever` rows) but not yet visualized — coverage/agreement trade-off plot.
- Bootstrap CIs / significance for the gate deltas (per-query scores not yet persisted).
- Design §9 decision log: battery + margins for RQ3, semantic backend go/no-go.